# Grant Lakebase Schema Access

Creates the `app` schema in a Lakebase database and grants the app's auto-provisioned
service principal full privileges, including `CREATE ON DATABASE` for schema management.

**Parameters:**
- `principal` — App SPN application_id (UUID / PG role name)
- `lakebase_project_id` — Lakebase project ID
- `lakebase_database_id` — Lakebase database resource ID
- `schema_name` — Schema to create and grant access on (default: `app`)

In [0]:
%pip install psycopg2-binary --upgrade
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("principal", "", "App SPN application_id (UUID)")
dbutils.widgets.text("lakebase_project_id", "", "Lakebase Project ID")
dbutils.widgets.text("lakebase_database_id", "", "Lakebase Database ID")
dbutils.widgets.text("schema_name", "app", "Schema to create and grant")

In [0]:
principal = dbutils.widgets.get("principal")
lakebase_project_id = dbutils.widgets.get("lakebase_project_id")
lakebase_database_id = dbutils.widgets.get("lakebase_database_id")
schema_name = dbutils.widgets.get("schema_name")

assert principal, "principal is required"
assert lakebase_project_id, "lakebase_project_id is required"
assert lakebase_database_id, "lakebase_database_id is required"
assert schema_name, "schema_name is required"

print(f"  Principal (PG role): {principal}")
print(f"  Lakebase project:   {lakebase_project_id}")
print(f"  Database ID:        {lakebase_database_id}")
print(f"  Schema:             {schema_name}")

In [0]:
import requests as req
import json as json_mod
from base64 import urlsafe_b64decode
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Resolve the Lakebase endpoint for the production branch
branch_path = f"projects/{lakebase_project_id}/branches/production"
auth_headers = w.config.authenticate()
host_url = w.config.host

resp = req.get(
    f"{host_url}/api/2.0/postgres/{branch_path}/endpoints",
    headers=auth_headers
)
resp.raise_for_status()
endpoints_data = resp.json().get("endpoints", [])

if not endpoints_data:
    raise RuntimeError(
        f"No endpoint found for '{branch_path}'. "
        f"The Lakebase project may still be initializing."
    )

ep = endpoints_data[0]
endpoint_name = ep["name"]
pg_host = ep["status"]["hosts"]["host"]
pg_port = 5432

print(f"  Endpoint:  {endpoint_name}")
print(f"  Host:      {pg_host}")
print(f"  Port:      {pg_port}")

# Generate a database credential
cred_resp = req.post(
    f"{host_url}/api/2.0/postgres/credentials",
    headers=auth_headers,
    json={"endpoint": endpoint_name}
)
cred_resp.raise_for_status()
cred = cred_resp.json()
pg_token = cred["token"]

# Extract PG username from the JWT 'sub' claim
parts = pg_token.split(".")
payload_b64 = parts[1] + "=" * (4 - len(parts[1]) % 4)
payload = json_mod.loads(urlsafe_b64decode(payload_b64))
pg_user = payload.get("sub", "")

print(f"  PG user:   {pg_user}")

In [0]:
import psycopg2

conn = psycopg2.connect(
    host=pg_host,
    port=pg_port,
    dbname="databricks_postgres",
    user=pg_user,
    password=pg_token,
    sslmode="require"
)
conn.autocommit = True
cur = conn.cursor()

# Verify connection
cur.execute("SELECT current_user, current_database()")
row = cur.fetchone()
print(f"  Connected: user={row[0]}, db={row[1]}")
db_name = row[1]

# List existing roles
cur.execute("SELECT rolname FROM pg_roles ORDER BY rolname")
roles = [r[0] for r in cur.fetchall()]
print(f"  Existing roles: {roles}")

# Determine the role name for the app SPN
role_name = principal
if principal not in roles:
    matching = [r for r in roles if principal in r]
    if matching:
        role_name = matching[0]
        print(f"  Matched role: {role_name}")
    else:
        print(f"  Warning: No role matching '{principal}' found in pg_roles.")
        print(f"  Will attempt GRANT to '{principal}' directly.")

print(f"  Target role: {role_name}")

# 1. Grant CREATE on the database (allows the SPN to create schemas)
cur.execute(f'GRANT CREATE ON DATABASE "{db_name}" TO "{role_name}"')
print(f"  \u2713 Granted CREATE ON DATABASE {db_name} TO \"{role_name}\"")

# 2. Create the app schema
cur.execute(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")
print(f"  \u2713 Schema '{schema_name}' created (or already exists)")

# 3. Grant ALL on the schema to the app SPN role
cur.execute(f'GRANT ALL ON SCHEMA {schema_name} TO "{role_name}"')
print(f"  \u2713 Granted ALL ON SCHEMA {schema_name} TO \"{role_name}\"")

# 4. Set default privileges
cur.execute(
    f'ALTER DEFAULT PRIVILEGES IN SCHEMA {schema_name} '
    f'GRANT ALL ON TABLES TO "{role_name}"'
)
print(f"  \u2713 Default table privileges set for \"{role_name}\"")

cur.close()
conn.close()
print("\n  Done. App SPN can CREATE schemas and manage objects in the 'app' schema.")

In [0]:
# Reconnect to verify
conn = psycopg2.connect(
    host=pg_host, port=pg_port, dbname="databricks_postgres",
    user=pg_user, password=pg_token, sslmode="require"
)
conn.autocommit = True
cur = conn.cursor()

cur.execute(f"SELECT has_schema_privilege('{role_name}', '{schema_name}', 'CREATE')")
has_schema_create = cur.fetchone()[0]

cur.execute(f"SELECT has_schema_privilege('{role_name}', '{schema_name}', 'USAGE')")
has_usage = cur.fetchone()[0]

cur.execute(f"SELECT has_database_privilege('{role_name}', current_database(), 'CREATE')")
has_db_create = cur.fetchone()[0]

print(f"  Verification for role '{role_name}':")
print(f"    CREATE ON DATABASE: {has_db_create}")
print(f"    CREATE ON SCHEMA:   {has_schema_create}")
print(f"    USAGE ON SCHEMA:    {has_usage}")

if has_db_create and has_schema_create and has_usage:
    print("\n  \u2713 All permissions verified.")
else:
    raise RuntimeError(
        f"Permission check failed: DB_CREATE={has_db_create}, "
        f"SCHEMA_CREATE={has_schema_create}, USAGE={has_usage}."
    )

cur.close()
conn.close()